In [1]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

FIGURE_DIR = PROJECT_ROOT / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (7, 4.5),
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False
})

def save_figure(fig, filename):
    fig.savefig(
        FIGURE_DIR / f"{filename}.png",
        dpi=300,
        bbox_inches="tight"
    )
    print(f"Saved: {filename}")
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "monthly_features_qc.csv"

data = pd.read_csv(DATA_PATH)
df = data.loc[data["ELIGIBLE_B_90"] == True].copy()

print("Full dataset:", data.shape)
print("Comparison observations:", len(df))


Full dataset: (2016, 40)
Comparison observations: 1768


In [3]:
station_df = df[df['STATION'].astype(str) == '41184099999'].copy()

station_df['MONTH_PERIOD'] = pd.to_datetime(station_df['MONTH_PERIOD'])
station_df = station_df.sort_values("MONTH_PERIOD")

station_df = station_df.sort_values("MONTH_PERIOD")

display(station_df[['MONTH_PERIOD','PRCP_TOTAL','PRCP_LAG_0','PRCP_LAG_1','PRCP_LAG_2','TARGET_NEXT_MONTH']].head(15))
print("PRCP_TOTAL equals PRCP_LAG_0:", np.allclose(df["PRCP_TOTAL"],df['PRCP_LAG_0']))

,MONTH_PERIOD,PRCP_TOTAL,PRCP_LAG_0,PRCP_LAG_1,PRCP_LAG_2,TARGET_NEXT_MONTH
2,1996-03-01,0.000,0.000,0.000,0.000,0.000
3,1996-04-01,0.000,0.000,0.000,0.000,0.000
4,1996-05-01,0.000,0.000,0.000,0.000,0.000
5,1996-06-01,0.000,0.000,0.000,0.000,0.000
6,1996-07-01,0.000,0.000,0.000,0.000,0.000
7,1996-08-01,0.000,0.000,0.000,0.000,0.000
8,1996-09-01,0.000,0.000,0.000,0.000,0.000
9,1996-10-01,0.000,0.000,0.000,0.000,0.000
10,1996-11-01,0.000,0.000,0.000,0.000,0.000
11,1996-12-01,0.000,0.000,0.000,0.000,0.000


PRCP_TOTAL equals PRCP_LAG_0: True


In [ ]:
calendar_df = data.copy()

calendar_df["STATION"] = calendar_df["STATION"].astype(str)
calendar_df["MONTH_PERIOD"] = pd.to_datetime(calendar_df["MONTH_PERIOD"])

calendar_df = calendar_df.sort_values(["STATION", "MONTH_PERIOD"])
duplicates = calendar_df.duplicated(["STATION", "MONTH_PERIOD"]).sum()

print("Duplicate station-month rows:", duplicates)

Duplicate station-month rows: 0


In [6]:
calendar_df["EXPECTED_NEXT_MONTH"] = (
    calendar_df.groupby("STATION")["PRCP_TOTAL"]
    .shift(-1)
)

calendar_df["EXPECTED_NEXT_DATE"] = (
    calendar_df["MONTH_PERIOD"]
    + pd.DateOffset(months=1)
)

calendar_df["ACTUAL_NEXT_DATE"] = (
    calendar_df.groupby("STATION")["MONTH_PERIOD"]
    .shift(-1)
)

consecutive_mask = (
    calendar_df["ACTUAL_NEXT_DATE"]
    == calendar_df["EXPECTED_NEXT_DATE"]
)

valid_mask = (
    consecutive_mask
    & calendar_df["TARGET_NEXT_MONTH"].notna()
    & calendar_df["EXPECTED_NEXT_MONTH"].notna()
)

matches = np.isclose(
    calendar_df.loc[valid_mask, "TARGET_NEXT_MONTH"],
    calendar_df.loc[valid_mask, "EXPECTED_NEXT_MONTH"]
)

print("Compared rows:", valid_mask.sum())
print("Mismatched targets:", (~matches).sum())


Compared rows: 2008
Mismatched targets: 0


In [7]:

excluded_df = calendar_df.loc[
    ~valid_mask,
    [
        "STATION",
        "MONTH_PERIOD",
        "PRCP_TOTAL",
        "TARGET_NEXT_MONTH",
        "EXPECTED_NEXT_MONTH",
        "ACTUAL_NEXT_DATE",
        "EXPECTED_NEXT_DATE"
    ]
].copy()

print("Excluded rows:", len(excluded_df))

display(excluded_df)


Excluded rows: 8


,STATION,MONTH_PERIOD,PRCP_TOTAL,TARGET_NEXT_MONTH,EXPECTED_NEXT_MONTH,ACTUAL_NEXT_DATE,EXPECTED_NEXT_DATE
119,41184099999,2005-12-01,0.000,NaN,5.08,2006-02-01,2006-01-01
335,41184099999,2023-12-01,60.706,NaN,NaN,NaT,2024-01-01
120,41184099999,NaT,NaN,5.08,NaN,NaT,NaT
671,41194099999,2023-12-01,0.000,NaN,NaN,NaT,2024-01-01
1007,41196099999,2023-12-01,0.000,NaN,NaN,NaT,2024-01-01
1343,41198099999,2023-12-01,0.000,NaN,NaN,NaT,2024-01-01
1679,41217099999,2023-12-01,0.000,NaN,NaN,NaT,2024-01-01
2015,41218099999,2023-12-01,0.000,NaN,NaN,NaT,2024-01-01


In [8]:

# Inspect original records around January 2006

display(
    data.loc[
        [119, 120, 121],
        [
            "STATION",
            "MONTH",
            "MONTH_PERIOD",
            "PRCP_TOTAL",
            "TARGET_NEXT_MONTH",
            "PRCP_LAG_0",
            "PRCP_LAG_1",
            "PRCP_LAG_2"
        ]
    ]
)

print("\nOriginal MONTH_PERIOD value:")
print(repr(data.loc[120, "MONTH_PERIOD"]))

print("\nOriginal MONTH value:")
print(repr(data.loc[120, "MONTH"]))


,STATION,MONTH,MONTH_PERIOD,PRCP_TOTAL,TARGET_NEXT_MONTH,PRCP_LAG_0,PRCP_LAG_1,PRCP_LAG_2
119,41184099999,2005-12,2005-12,0.00,NaN,0.00,0.0,0.0
120,41184099999,2006-01,NaN,NaN,5.08,NaN,0.0,0.0
121,41184099999,2006-02,2006-02,5.08,0.00,5.08,NaN,0.0



Original MONTH_PERIOD value:
nan

Original MONTH value:
'2006-01'


In [9]:

calendar_check = data.copy()

calendar_check["STATION"] = (
    calendar_check["STATION"].astype(str)
)

calendar_check["MONTH_DATE"] = pd.to_datetime(
    calendar_check["MONTH"],
    format="%Y-%m"
)

print(
    "Missing MONTH_DATE:",
    calendar_check["MONTH_DATE"].isna().sum()
)

print(
    "Duplicate station-month:",
    calendar_check.duplicated(
        ["STATION", "MONTH_DATE"]
    ).sum()
)

print(
    "January 2006 date:",
    calendar_check.loc[120, "MONTH_DATE"]
)


Missing MONTH_DATE: 0
Duplicate station-month: 0
January 2006 date: 2006-01-01 00:00:00


In [12]:
calendar_check = calendar_check.sort_values(
    ["STATION", "MONTH_DATE"]
).copy()

# Calculate next calendar month's actual rainfall
calendar_check["EXPECTED_TARGET"] = (
    calendar_check.groupby("STATION")["PRCP_TOTAL"]
    .shift(-1)
)

# Calculate next row's date
calendar_check["NEXT_DATE"] = (
    calendar_check.groupby("STATION")["MONTH_DATE"]
    .shift(-1)
)

# Expected calendar date
calendar_check["EXPECTED_DATE"] = (
    calendar_check["MONTH_DATE"]
    + pd.DateOffset(months=1)
)

print(
    "Non-consecutive months:",
    (
        calendar_check["NEXT_DATE"].notna()
        & (calendar_check["NEXT_DATE"] !=
           calendar_check["EXPECTED_DATE"])
    ).sum()
)


valid_mask = (
    calendar_check["TARGET_NEXT_MONTH"].notna()
    & calendar_check["EXPECTED_TARGET"].notna()
    & (calendar_check["NEXT_DATE"] == calendar_check["EXPECTED_DATE"])
)

matches = np.isclose(
    calendar_check.loc[valid_mask, "TARGET_NEXT_MONTH"],
    calendar_check.loc[valid_mask, "EXPECTED_TARGET"]
)

print("Compared rows:", valid_mask.sum())
print("Mismatched targets:", (~matches).sum())

missing_agreement = (
    calendar_check["TARGET_NEXT_MONTH"].isna()
    == calendar_check["EXPECTED_TARGET"].isna()
)

print(
    "Missing-status disagreements:",
    (~missing_agreement).sum()
)



Non-consecutive months: 0
Compared rows: 2009
Mismatched targets: 0
Missing-status disagreements: 0


In [14]:
# Current month rainfall
calendar_check["EXPECTED_LAG_0"] = (
    calendar_check["PRCP_TOTAL"]
)

# Previous month rainfall
calendar_check["EXPECTED_LAG_1"] = (
    calendar_check.groupby("STATION")["PRCP_TOTAL"]
    .shift(1)
)

# Rainfall two months ago
calendar_check["EXPECTED_LAG_2"] = (
    calendar_check.groupby("STATION")["PRCP_TOTAL"]
    .shift(2)
)

for lag in [0, 1, 2]:

    actual_col = f"PRCP_LAG_{lag}"
    expected_col = f"EXPECTED_LAG_{lag}"

    matches = np.isclose(
        calendar_check[actual_col],
        calendar_check[expected_col],
        equal_nan=True
    )

    print(
        f"LAG_{lag} mismatches:",
        (~matches).sum()
    )


LAG_0 mismatches: 0
LAG_1 mismatches: 0
LAG_2 mismatches: 0


In [16]:
seasonal_df = df.copy()

seasonal_df['OBS_DATE'] = pd.to_datetime(seasonal_df['MONTH'], format="%Y-%m")
seasonal_df['TARGET_DATE'] = (seasonal_df['OBS_DATE'] + pd.DateOffset(months=1))
seasonal_df['TARGET_MONTH'] = (seasonal_df['TARGET_DATE'].dt.month)

# MONTH_SIN = sin( 2(pi)m / 12)
# MONTH_COS = cos( 2(pi)m / 12)
seasonal_df['MONTH_SIN'] = np.sin( 2 * np.pi * seasonal_df['TARGET_MONTH'] / 12)
seasonal_df['MONTH_COS'] = np.cos( 2 * np.pi * seasonal_df['TARGET_MONTH'] / 12)

display(
    seasonal_df[['MONTH','TARGET_DATE','TARGET_MONTH','MONTH_SIN','MONTH_COS']]
    .head(12)
)

,MONTH,TARGET_DATE,TARGET_MONTH,MONTH_SIN,MONTH_COS
2,1996-03,1996-04-01,4,8.660254e-01,-5.000000e-01
3,1996-04,1996-05-01,5,5.000000e-01,-8.660254e-01
4,1996-05,1996-06-01,6,1.224647e-16,-1.000000e+00
5,1996-06,1996-07-01,7,-5.000000e-01,-8.660254e-01
6,1996-07,1996-08-01,8,-8.660254e-01,-5.000000e-01
7,1996-08,1996-09-01,9,-1.000000e+00,-1.836970e-16
8,1996-09,1996-10-01,10,-8.660254e-01,5.000000e-01
9,1996-10,1996-11-01,11,-5.000000e-01,8.660254e-01
10,1996-11,1996-12-01,12,-2.449294e-16,1.000000e+00
11,1996-12,1997-01-01,1,5.000000e-01,8.660254e-01


In [18]:
weather_features = [
    "TEMP_MEAN",
    "DEWP_MEAN",
    "WDSP_MEAN",
    "TEMP_MIN_MEAN",
    "TEMP_MAX_MEAN"
]

audit_df = seasonal_df[
    ["STATION", "MONTH", "TARGET_DATE"]
    + weather_features
].copy()

# Select one station
audit_df = audit_df[
    audit_df["STATION"].astype(str) == "41184099999"
]

display(audit_df.head(12))

print("\nMissing weather values:")
print(audit_df[weather_features].isna().sum())


,STATION,MONTH,TARGET_DATE,TEMP_MEAN,DEWP_MEAN,WDSP_MEAN,TEMP_MIN_MEAN,TEMP_MAX_MEAN
2,41184099999,1996-03,1996-04-01,22.724014,16.976703,8.602839,17.258065,28.840502
3,41184099999,1996-04,1996-05-01,27.048148,16.362963,7.864827,18.666667,35.477778
4,41184099999,1996-05,1996-06-01,31.620072,18.120072,7.234748,21.700717,40.962366
5,41184099999,1996-06,1996-07-01,34.922222,21.587037,9.562493,27.151852,43.170370
6,41184099999,1996-07,1996-08-01,36.951613,20.331541,9.170387,27.894265,45.318996
7,41184099999,1996-08,1996-09-01,35.923372,24.113027,8.915145,29.344828,43.339080
8,41184099999,1996-09,1996-10-01,32.970370,23.579630,7.864827,26.068519,40.664815
9,41184099999,1996-10,1996-11-01,28.103943,17.967742,6.828503,19.580645,36.783154
10,41184099999,1996-11,1996-12-01,23.557407,13.783333,8.105587,15.746296,31.442593
11,41184099999,1996-12,1997-01-01,19.263441,11.768817,5.818865,11.241935,27.555556



Missing weather values:
TEMP_MEAN        0
DEWP_MEAN        0
WDSP_MEAN        0
TEMP_MIN_MEAN    0
TEMP_MAX_MEAN    0
dtype: int64


In [20]:
min_violations = (
    seasonal_df["TEMP_MIN_MEAN"]
    > seasonal_df["TEMP_MEAN"]
)

max_violations = (
    seasonal_df["TEMP_MEAN"]
    > seasonal_df["TEMP_MAX_MEAN"]
)

print("Minimum temperature violations:", min_violations.sum())
print("Maximum temperature violations:", max_violations.sum())

print("\nForecast date alignment:")
print(
    (
        seasonal_df["TARGET_DATE"]
        == seasonal_df["OBS_DATE"] + pd.DateOffset(months=1)
    ).all()
)


Minimum temperature violations: 0
Maximum temperature violations: 0

Forecast date alignment:
True


In [24]:
features_A = [
    "TEMP_MEAN",
    "DEWP_MEAN",
    "WDSP_MEAN",
    "TEMP_MIN_MEAN",
    "TEMP_MAX_MEAN",
    "MONTH_COS",
    "MONTH_SIN"
]

features_B = features_A + [
    "PRCP_LAG_0",
    "PRCP_LAG_1",
    "PRCP_LAG_2"
]

OUTPUT_PATH = (PROJECT_ROOT / 'data' / 'processed' / 'monthly_features_seasonal.csv')

seasonal_df.to_csv(OUTPUT_PATH, index= False)
saved_df = pd.read_csv(OUTPUT_PATH)
print("Saved dataset:", OUTPUT_PATH)
print("Saved shape:", saved_df.shape)

Saved dataset: c:\Users\marya\Desktop\rainfall_project\data\processed\monthly_features_seasonal.csv
Saved shape: (1768, 45)
